In [ ]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 1.4 MB/s eta 0:00:00


# Load data
[link](https://zindi.africa/competitions/dataorg-financial-health-prediction-challenge/data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv('Train.csv')

In [ ]:
train = df.copy()

In [ ]:
train.head()

,ID,country,owner_age,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,personal_income,business_expenses,...,has_internet_banking,has_debit_card,future_risk_theft_stock,business_age_months,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,Target
0,ID_3CFL0U,eswatini,63.0,Yes,No,No,No,Yes,3000.0,6000.0,...,Never had,Never had,NaN,6.0,Never had,Used to have but don’t have now,NaN,Never had,Never had,Low
1,ID_XWI7G3,zimbabwe,39.0,No,Yes,Yes,No,Yes,NaN,NaN,...,NaN,NaN,No,3.0,Never had,Never had,NaN,NaN,NaN,Medium
2,ID_TY93LV,malawi,34.0,Don’t know or N/A,No,No,Don't know,Yes,30000.0,6000.0,...,Never had,Never had,Yes,NaN,NaN,NaN,Yes,NaN,NaN,Low
3,ID_9OP2C8,malawi,28.0,Yes,No,No,No,No,180000.0,60000.0,...,Never had,Never had,No,NaN,NaN,NaN,Yes,Never had,Have now,Low
4,ID_13REYS,zimbabwe,43.0,Yes,No,No,Yes,Yes,50.0,2400.0,...,NaN,NaN,No,0.0,Never had,Never had,Yes,NaN,NaN,Low


In [ ]:
train = train.drop(['ID'], axis=1)

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9618 entries, 0 to 9617
Data columns (total 38 columns):
 #   Column                                                            Non-Null Count  Dtype  
---  ------                                                            --------------  -----  
 0   country                                                           9618 non-null   object 
 1   owner_age                                                         9618 non-null   float64
 2   attitude_stable_business_environment                              9616 non-null   object 
 3   attitude_worried_shutdown                                         9616 non-null   object 
 4   compliance_income_tax                                             9614 non-null   object 
 5   perception_insurance_doesnt_cover_losses                          9613 non-null   object 
 6   perception_cannot_afford_insurance                                9613 non-null   object 
 7   personal_income                  

In [ ]:
train.shape

(9618, 38)

# **Missing values**



In [ ]:
train.head()

In [ ]:
train.isnull().sum().sort_values(ascending=False)

In [ ]:
categorical_cols = train.select_dtypes(include=['object']).columns

train[categorical_cols] = train[categorical_cols].fillna('Missing')

In [ ]:
numerical_cols_to_fill = ['personal_income', 'business_expenses', 'business_turnover', 'business_age_years', 'business_age_months']

for col in numerical_cols_to_fill:
    if train[col].isnull().sum() > 0:
        mean_val = train[col].mean()
        train[col] = train[col].fillna(mean_val)

print("Missing values after filling numerical columns:")
print(train[numerical_cols_to_fill].isnull().sum())

In [ ]:
train.isnull().sum().sort_values(ascending=False)

In [ ]:
train.head()

In [ ]:
print("Final check for missing values in the entire DataFrame:")
print(train.isnull().sum().sort_values(ascending=False))

# **Encode**

In [ ]:
target_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
train['Target'] = train['Target'].map(target_mapping)

# Encode ordinal 'Had' status columns
had_status_cols = [
    'motor_vehicle_insurance',
    'has_credit_card',
    'has_internet_banking',
    'has_mobile_money',
    'has_loan_account',
    'has_debit_card'
]
had_status_mapping = {'Never had': 0, 'Used to have but don’t have now': 1, 'Have now': 2}
for col in had_status_cols:
    train[col] = train[col].map(had_status_mapping)

# Encode remaining categorical columns that still contain 'Missing' strings
remaining_had_status_cols = [
    'medical_insurance',
    'funeral_insurance',
    'uses_friends_family_savings',
    'uses_informal_lender'
]
for col in remaining_had_status_cols:
    train[col] = train[col].map(had_status_mapping)

remaining_binary_map_cols = [
    'future_risk_theft_stock',
    'motivation_make_more_money'
]
binary_mapping = {'Yes': 1, 'No': 0}
for col in remaining_binary_map_cols:
    train[col] = train[col].map(binary_mapping)


In [ ]:
train.head()

,country,owner_age,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,personal_income,business_expenses,business_turnover,...,has_internet_banking,has_debit_card,future_risk_theft_stock,business_age_months,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,Target
0,eswatini,63.0,Yes,No,No,No,Yes,3000.0,6000.0,7000.0,...,0.0,0.0,NaN,6.0,0.0,1.0,NaN,0.0,0.0,0
1,zimbabwe,39.0,No,Yes,Yes,No,Yes,NaN,NaN,NaN,...,NaN,NaN,0.0,3.0,0.0,0.0,NaN,NaN,NaN,1
2,malawi,34.0,Don’t know or N/A,No,No,Don't know,Yes,30000.0,6000.0,13000.0,...,0.0,0.0,1.0,NaN,NaN,NaN,1.0,NaN,NaN,0
3,malawi,28.0,Yes,No,No,No,No,180000.0,60000.0,30000.0,...,0.0,0.0,0.0,NaN,NaN,NaN,1.0,0.0,2.0,0
4,zimbabwe,43.0,Yes,No,No,Yes,Yes,50.0,2400.0,1800.0,...,NaN,NaN,0.0,0.0,0.0,0.0,1.0,NaN,NaN,0


In [ ]:
binary_map_cols = [
    'attitude_stable_business_environment',
    'attitude_worried_shutdown',
    'compliance_income_tax',
    'perception_insurance_doesnt_cover_losses',
    'perception_cannot_afford_insurance',

    'current_problem_cash_flow',
    'has_cellphone',
    'offers_credit_to_customers',
    'attitude_satisfied_with_achievement',
    'keeps_financial_records',
    'perception_insurance_companies_dont_insure_businesses_like_yours',

    'owner_sex',

    'perception_insurance_important',
    'has_insurance',
    'covid_essential_service',
    'attitude_more_successful_next_year',
    'problem_sourcing_money',
    'marketing_word_of_mouth',
]
binary_mapping = {'Yes': 1, 'No': 0, 'Don’t know or N/A': -1}
for col in binary_map_cols:
    train[col] = train[col].map(binary_mapping)

In [ ]:
one_hot_cols = ['country']
train = pd.get_dummies(train, columns=one_hot_cols, drop_first=True)

In [ ]:
train.head()

,owner_age,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,personal_income,business_expenses,business_turnover,business_age_years,...,business_age_months,medical_insurance,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,Target,country_lesotho,country_malawi,country_zimbabwe
0,63.0,1.0,0.0,0.0,0.0,1.0,3000.0,6000.0,7000.0,14.0,...,6.0,0.0,1.0,NaN,0.0,0.0,0,False,False,False
1,39.0,0.0,1.0,1.0,0.0,1.0,NaN,NaN,NaN,15.0,...,3.0,0.0,0.0,NaN,NaN,NaN,1,False,False,True
2,34.0,-1.0,0.0,0.0,NaN,1.0,30000.0,6000.0,13000.0,5.0,...,NaN,NaN,NaN,1.0,NaN,NaN,0,False,True,False
3,28.0,1.0,0.0,0.0,0.0,0.0,180000.0,60000.0,30000.0,1.0,...,NaN,NaN,NaN,1.0,0.0,2.0,0,False,True,False
4,43.0,1.0,0.0,0.0,1.0,1.0,50.0,2400.0,1800.0,3.0,...,0.0,0.0,0.0,1.0,NaN,NaN,0,False,False,True


In [ ]:
train.shape

(9618, 40)

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9618 entries, 0 to 9617
Data columns (total 40 columns):
 #   Column                                                            Non-Null Count  Dtype  
---  ------                                                            --------------  -----  
 0   owner_age                                                         9618 non-null   float64
 1   attitude_stable_business_environment                              9616 non-null   float64
 2   attitude_worried_shutdown                                         9616 non-null   float64
 3   compliance_income_tax                                             9262 non-null   float64
 4   perception_insurance_doesnt_cover_losses                          7090 non-null   float64
 5   perception_cannot_afford_insurance                                8149 non-null   float64
 6   personal_income                                                   9509 non-null   float64
 7   business_expenses                

In [ ]:
train.isnull().sum().sort_values(ascending=False)

,0
owner_sex,9618
offers_credit_to_customers,6888
uses_informal_lender,6233
uses_friends_family_savings,5889
keeps_financial_records,5145
current_problem_cash_flow,5043
has_loan_account,4499
medical_insurance,4403
funeral_insurance,4362
has_internet_banking,4329


In [ ]:
train.shape

(9618, 40)

In [ ]:
df.shape

(9618, 39)

# Visal

# **Split**

In [ ]:
# Create squared features in train DataFrame
train['funeral_insurance_sq'] = train['funeral_insurance']**2
train['has_credit_card_sq'] = train['has_credit_card']**2

In [ ]:
train.head()

,owner_age,attitude_stable_business_environment,attitude_worried_shutdown,compliance_income_tax,perception_insurance_doesnt_cover_losses,perception_cannot_afford_insurance,personal_income,business_expenses,business_turnover,business_age_years,...,funeral_insurance,motivation_make_more_money,uses_friends_family_savings,uses_informal_lender,Target,country_lesotho,country_malawi,country_zimbabwe,funeral_insurance_sq,has_credit_card_sq
0,63.0,1.0,0.0,0.0,0.0,1.0,3000.0,6000.0,7000.0,14.0,...,1.0,NaN,0.0,0.0,0,False,False,False,1.0,0.0
1,39.0,0.0,1.0,1.0,0.0,1.0,NaN,NaN,NaN,15.0,...,0.0,NaN,NaN,NaN,1,False,False,True,0.0,0.0
2,34.0,-1.0,0.0,0.0,NaN,1.0,30000.0,6000.0,13000.0,5.0,...,NaN,1.0,NaN,NaN,0,False,True,False,NaN,0.0
3,28.0,1.0,0.0,0.0,0.0,0.0,180000.0,60000.0,30000.0,1.0,...,NaN,1.0,0.0,2.0,0,False,True,False,NaN,0.0
4,43.0,1.0,0.0,0.0,1.0,1.0,50.0,2400.0,1800.0,3.0,...,0.0,1.0,NaN,NaN,0,False,False,True,0.0,0.0


In [ ]:
train.shape

(9618, 42)

In [ ]:
X = train.drop('Target', axis=1)
y = train['Target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
def change_object_to_cat(train):
  # changes objects columns to category and returns dataframe and list

  train = train.copy()
  list_str_obj_cols = train.columns[train.dtypes == "object"].tolist()
  for str_obj_col in list_str_obj_cols:
      train[str_obj_col] = train[str_obj_col].astype("category")

  return train,list_str_obj_cols
X, cat_list = change_object_to_cat(X)
X_test, cat_list = change_object_to_cat(X_val)

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [ ]:
model_xgb = XGBClassifier(random_state=42, eval_metric='mlogloss')

In [ ]:
model_xgb.fit(X_train_scaled, y_train)

y_predt = model_xgb.predict(X_val_scaled)
print("\nClassification Report for XGBoost Classifier:")
print(classification_report(y_val, y_predt))
print("\nAccuracy Score for XGBoost Classifier:", accuracy_score(y_val, y_predt))


Classification Report for XGBoost Classifier:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91      1544
           1       0.80      0.72      0.76       735
           2       0.83      0.51      0.63       126

    accuracy                           0.86      2405
   macro avg       0.84      0.73      0.77      2405
weighted avg       0.85      0.86      0.85      2405


Accuracy Score for XGBoost Classifier: 0.8557172557172558


In [ ]:
# 1. Calculate class weights for the y_train dataset
class_counts = pd.Series(y_train).value_counts()
total_samples = len(y_train)
n_classes = len(class_counts)

class_weights = {}
for cls in class_counts.index:
    class_weights[cls] = total_samples / (n_classes * class_counts[cls])

print("Calculated Class Weights:", class_weights)

# 2. Create a sample_weight array corresponding to y_train
sample_weights = y_train.map(class_weights)

# 3. Initialize a new XGBClassifier instance
model_xgb_weighted = XGBClassifier(random_state=42, learning_rate=0.1, max_depth=7, n_estimators=200, subsample=0.8, eval_metric='mlogloss')

# 4. Train this new XGBClassifier on X_train and y_train with sample_weight
model_xgb_weighted.fit(X_train_scaled, y_train, sample_weight=sample_weights)

# 5. Make predictions on the validation set X_val
y_pred_xgb_weighted = model_xgb_weighted.predict(X_val_scaled)

# 6. Print the classification report and accuracy score
print(
    "\nClassification Report for XGBoost Classifier (with Class Weights):")
print(classification_report(y_val, y_pred_xgb_weighted))
print(
    "\nAccuracy Score for XGBoost Classifier (with Class Weights):",
    accuracy_score(y_val, y_pred_xgb_weighted),)

Calculated Class Weights: {0: np.float64(0.5076717342342343), 1: np.float64(1.1272073761525239), 2: np.float64(6.989341085271318)}

Classification Report for XGBoost Classifier (with Class Weights):
              precision    recall  f1-score   support

           0       0.90      0.93      0.91      1544
           1       0.77      0.76      0.77       735
           2       0.80      0.57      0.67       126

    accuracy                           0.86      2405
   macro avg       0.82      0.75      0.78      2405
weighted avg       0.85      0.86      0.85      2405


Accuracy Score for XGBoost Classifier (with Class Weights): 0.8565488565488566


In [ ]:
model_xgb_weighted = XGBClassifier(random_state=42, learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.7, eval_metric='mlogloss')

# 4. Train this new XGBClassifier on X_train and y_train with sample_weight
model_xgb_weighted.fit(X_train_scaled, y_train, sample_weight=sample_weights)

# 5. Make predictions on the validation set X_val
y_pred_xgb_weighted = model_xgb_weighted.predict(X_val_scaled)

# 6. Print the classification report and accuracy score
print(
    "\nClassification Report for XGBoost Classifier (with Class Weights):"
)
print(classification_report(y_val, y_pred_xgb_weighted))
print(
    "\nAccuracy Score for XGBoost Classifier (with Class Weights):",
    accuracy_score(y_val, y_pred_xgb_weighted),
)


Classification Report for XGBoost Classifier (with Class Weights):
              precision    recall  f1-score   support

           0       0.91      0.88      0.90      1544
           1       0.73      0.76      0.74       735
           2       0.62      0.79      0.69       126

    accuracy                           0.84      2405
   macro avg       0.76      0.81      0.78      2405
weighted avg       0.84      0.84      0.84      2405


Accuracy Score for XGBoost Classifier (with Class Weights): 0.8382536382536383


In [ ]:
import joblib

# Define the filename for your model
model_filename = 'xgboost_model.pkl'

# Save the trained model to a file
joblib.dump(model_xgb_weighted, model_filename)

print(f"Model saved to {model_filename}")

Model saved to xgboost_model.pkl


# Address Class Imbalance with Resampling

In [ ]:
# First, install the imbalanced-learn library if you haven't already
!pip install imbalanced-learn

from imblearn.over_sampling import SMOTE

# Instantiate SMOTE
sm = SMOTE(random_state=42)

# Apply SMOTE to the training data
X_train_resampled, y_train_resampled = sm.fit_resample(X_train, y_train)

print("Original y_train shape:", y_train.shape)
print("Resampled y_train shape:", y_train_resampled.shape)
print("\nOriginal y_train class distribution:\n", pd.Series(y_train).value_counts())
print("\nResampled y_train class distribution:\n", pd.Series(y_train_resampled).value_counts())

In [ ]:
X_train_resampled.head()

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train_resampled)
X_train_scaledrs = scaler.transform(X_train_resampled)

# Sav

In [ ]:
import joblib

# Define the filename for your model
model_filename = 'xgboost_model.pkl'

# Save the trained model to a file
joblib.dump(model_xgb, model_filename)

print(f"Model saved to {model_filename}")

### Load the Model in Another Project

To use this model in a different project or at a later time, you can load it back from the saved file:

In [ ]:
import joblib

# Define the filename where the model is saved
model_filename = 'xgboost_model.pkl'

# Load the model from the file
loaded_model = joblib.load(model_filename)

print(f"Model loaded from {model_filename}")

# You can now use the loaded_model for predictions
# For example, to make predictions on your validation set:
# loaded_predictions = loaded_model.predict(X_val_scaled)
# print("Predictions from loaded model:", loaded_predictions[:5])

# Task
Perform hyperparameter tuning for an XGBoost Classifier using `GridSearchCV` to find the best `learning_rate`, `max_depth`, `n_estimators`, and `subsample`. The search should use 3-fold cross-validation and optimize for the `f1_weighted` score, with progress updates enabled. After fitting, print the best parameters and corresponding F1-weighted score. Finally, evaluate the best model's performance on the validation set (`X_val_scaled`, `y_val`) by printing the classification report and accuracy score.

## Define Hyperparameter Grid

### Subtask:
Define a dictionary with hyperparameter ranges for `learning_rate`, `max_depth`, `n_estimators`, and `subsample`.


**Reasoning**:
To define the hyperparameter grid as requested in the subtask, I will create a Python dictionary named `param_grid` with the specified ranges for `learning_rate`, `max_depth`, `n_estimators`, and `subsample`.



In [ ]:
param_grid = {
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
}

print("Hyperparameter grid defined:")
print(param_grid)

Hyperparameter grid defined:
{'learning_rate': [0.01, 0.1, 0.2], 'max_depth': [3, 5, 7], 'n_estimators': [100, 200, 300], 'subsample': [0.7, 0.8, 0.9], 'min_child_weight': [1, 3, 5], 'gamma': [0, 0.1, 0.2]}


## Initialize GridSearchCV

### Subtask:
Set up GridSearchCV with the XGBClassifier model, the defined hyperparameter grid, cv=3 for cross-validation, and scoring='f1_weighted' to optimize for F1-score. Also, include verbose=1 for progress updates.


**Reasoning**:
To initialize GridSearchCV, I will import the necessary class, create an XGBClassifier instance, and then set up GridSearchCV with the specified parameters including the hyperparameter grid, cross-validation folds, scoring metric, and verbosity.



In [ ]:
from sklearn.model_selection import GridSearchCV

# Create an instance of XGBClassifier
model_xgb_grid = XGBClassifier(random_state=42, eval_metric='mlogloss')

# Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=model_xgb_grid,
    param_grid=param_grid,
    cv=3,
    scoring='f1_weighted',
    verbose=1
)

print("GridSearchCV initialized successfully.")

GridSearchCV initialized successfully.


## Fit GridSearchCV to Training Data

### Subtask:
Run the `fit` method of `GridSearchCV` on the scaled training data (`X_train_scaled`, `y_train`) to perform the hyperparameter search.


**Reasoning**:
To perform the hyperparameter search, I will call the `.fit()` method on the initialized `grid_search` object, providing the scaled training features `X_train_scaled` and the training target variable `y_train`.



In [ ]:
grid_search.fit(X_train_scaled, y_train)
print("GridSearchCV fitting complete.")

Fitting 3 folds for each of 729 candidates, totalling 2187 fits
GridSearchCV fitting complete.


## Print Best Parameters and Score

### Subtask:
Display the best hyperparameters found by `GridSearchCV` using `best_params_` and the corresponding best score using `best_score_`.


**Reasoning**:
To fulfill the subtask, I will access and print the `best_params_` and `best_score_` attributes from the `grid_search` object, which contains the results of the hyperparameter tuning.



In [ ]:
print("Best Parameters found by GridSearchCV:", grid_search.best_params_)
print("Best F1-weighted Score:", grid_search.best_score_)

Best Parameters found by GridSearchCV: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best F1-weighted Score: 0.8742392612142779


In [ ]:
print("Best Parameters found by GridSearchCV:", grid_search.best_params_)
print("Best F1-weighted Score:", grid_search.best_score_)

Best Parameters found by GridSearchCV: {'gamma': 0.2, 'learning_rate': 0.1, 'max_depth': 5, 'min_child_weight': 5, 'n_estimators': 100, 'subsample': 0.8}
Best F1-weighted Score: 0.8773516489250919


## Evaluate Best Model

### Subtask:
Use the `best_estimator_` from `GridSearchCV` to make predictions on the validation set (`X_val_scaled`) and print the classification report and accuracy score to assess its performance.


In [ ]:
model_xgb_weightedtun = XGBClassifier(random_state=42, learning_rate=0.1, max_depth=5,
                                   n_estimators=100, subsample= 0.8, eval_metric='mlogloss',
                                   gamma=0.2, min_child_weight=5)

# 4. Train this new XGBClassifier on X_train and y_train with sample_weight
model_xgb_weightedtun.fit(X_train_scaled, y_train, sample_weight=sample_weights)

# 5. Make predictions on the validation set X_val
y_pred_xgb_weighted = model_xgb_weightedtun.predict(X_val_scaled)

# 6. Print the classification report and accuracy score
print(
    "\nClassification Report for XGBoost Classifier (with Class Weights):"
)
print(classification_report(y_val, y_pred_xgb_weighted))
print(
    "\nAccuracy Score for XGBoost Classifier (with Class Weights):",
    accuracy_score(y_val, y_pred_xgb_weighted),
)


Classification Report for XGBoost Classifier (with Class Weights):
              precision    recall  f1-score   support

           0       0.92      0.88      0.90      1544
           1       0.73      0.79      0.76       735
           2       0.67      0.73      0.70       126

    accuracy                           0.84      2405
   macro avg       0.77      0.80      0.78      2405
weighted avg       0.85      0.84      0.84      2405


Accuracy Score for XGBoost Classifier (with Class Weights): 0.8415800415800416


In [ ]:
# 2. Initialize a new XGBClassifier instance with best parameters and class weights
model_xgb_tuned_weighted = XGBClassifier(
    random_state=42, learning_rate=0.4, max_depth=4, n_estimators=100, subsample=0.7,
    eval_metric='mlogloss', objective='multi:softmax',  num_class=3,
    booster='gbtree', reg_lambda=1.0,

    )

# 3. Train the XGBClassifier model on X_train and y_train with sample_weights
model_xgb_tuned_weighted.fit(X_train_scaled, y_train, sample_weight=sample_weights)

# 4. Make predictions on the validation set X_val
y_pred_tuned_weighted = model_xgb_tuned_weighted.predict(X_val_scaled)

# 5. Print the classification report and accuracy score
print(
    "\nClassification Report for Tuned XGBoost Classifier (with Class Weights):"
)
print(classification_report(y_val, y_pred_tuned_weighted))
print(
    "\nAccuracy Score for Tuned XGBoost Classifier (with Class Weights):",
    accuracy_score(y_val, y_pred_tuned_weighted),
)


Classification Report for Tuned XGBoost Classifier (with Class Weights):
              precision    recall  f1-score   support

           0       0.90      0.90      0.90      1544
           1       0.75      0.76      0.76       735
           2       0.75      0.67      0.71       126

    accuracy                           0.85      2405
   macro avg       0.80      0.78      0.79      2405
weighted avg       0.85      0.85      0.85      2405


Accuracy Score for Tuned XGBoost Classifier (with Class Weights): 0.8482328482328483


In [ ]:
import joblib

# Define the filename for your model
model_filename = 'xgboost_model.pkl'

# Save the trained model to a file
joblib.dump(model_xgb_tuned_weighted, model_filename)

print(f"Model saved to {model_filename}")

Model saved to xgboost_model.pkl
